In [3]:
# https://chatgpt.com/c/68bc2472-baec-8331-88cd-1410188ed832

# Patobulintas variantas su išsamesne dokumentacija ir klaidų tvarkymu.

"""
EKG triukšmų analizė su U-Net (Keras) modeliu.
- Patikrina modelio/žurnalų suderinamumą.
- Užkrauna modelį ir įrašų sąrašą iš Excel (ignoruoja tag == 9999).
- Filtruoja signalą, randa outliers/rdropouts, U-Net triukšmus.
- Skaičiuoja metrikas ir išsaugo .txt + .xlsx + summary.txt.

Reikalingi moduliai iš TRIUKSMU_DETEKTAVIMAS:
- zive_util_ml.get_ecg_signal
- use_ecg_denoising_util.{bandpass_filter, find_outliers_rdropouts, merge_lists_of_tuples, run_ecg_denoising_pipeline}
"""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import keras  # type: ignore
import numpy as np
import pandas as pd
import sys

from ecg_denoising_util import (
    get_ecg_signal, 
    ecg_filter,
    find_outliers_rdropouts,
    merge_lists_of_tuples,
    run_ecg_denoising_pipeline,
)


# === Konfigūracija ==============================================================
FS = 200
SEGMENT_LENGTH = 1024
OVERLAP = 0.5
CONFIG = {"FS": FS, "SEGMENT_LENGTH": SEGMENT_LENGTH, "OVERLAP": OVERLAP}

THRESHOLD = 0.08  # U-Net residual noise threshold

# ECG įrašo filtravimui
fp = {  'type': 'lowpass',
            'method':'butterworth',
            'order':5,
            'sampling_rate':FS,
            'lowcut':0.5,
            'highcut':90 }

DUOMENU_APLANKAS = Path.home() / "DI/2025_ZIVEO/DUOMENYS_UPD"
REZULTATU_APLANKAS = DUOMENU_APLANKAS / "REZULTATAI"
PARAMETRU_APLANKAS = DUOMENU_APLANKAS / "PARAMETRAI"
REC_DIR = DUOMENU_APLANKAS / "records_npy_all"

MODEL_FILE_NAME = "resunet_ecg_1024_0_5_3_6.keras"
# MARKER = MODEL_FILE_NAME.removeprefix("resunet_ecg").removesuffix(".keras")  + '_test' # -> "_1024_0_5_3_6_test"
MARKER = MODEL_FILE_NAME.removeprefix("resunet_ecg").removesuffix(".keras")  # -> "_1024_0_5_3_6"
NOISE_LOG_BASENAME = f"noise_log{MARKER}"

EXCEL_NAME = "visi_zive_irasai_atrankai.xlsx"
# EXCEL_NAME = "visi_zive_irasai_atrankai_test.xlsx"

# === Pagalbinės funkcijos =======================================================


def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )


def check_consistency(model_file_name: str, noise_log_basename: str, config: dict) -> None:
    """Patikrina failų sufiksų ir parametrų suderinamumą."""
    model_suffix = model_file_name.split("resunet_ecg")[-1].replace(".keras", "")
    noise_suffix = noise_log_basename.split("noise_log")[-1]

    # if model_suffix != noise_suffix:
    #     raise ValueError(f"Suffix mismatch:\n  model: {model_suffix}\n  noise: {noise_suffix}")

    seg_str = str(config["SEGMENT_LENGTH"])
    if seg_str not in model_suffix:
        raise ValueError(f"SEGMENT_LENGTH mismatch: expected '{seg_str}' in '{model_suffix}'")

    overlap_str = str(config["OVERLAP"]).replace(".", "_")
    if overlap_str not in model_suffix:
        raise ValueError(f"OVERLAP mismatch: expected '{overlap_str}' in '{model_suffix}'")

    logging.info("Suderinamumas OK | sufiksas:%s | SEGMENT_LENGTH:%s | OVERLAP:%s",
                 model_suffix, seg_str, overlap_str)


def load_model_checked(model_path: Path, segment_len: int) -> keras.Model:
    """Užkrauna Keras modelį ir patikrina input shape == (segment_len, 1)."""
    logging.info("Kraunamas modelis: %s", model_path)
    try:
        model = keras.models.load_model(str(model_path))
    except Exception as exc:  # noqa: BLE001
        raise ValueError(f"Failed to load model from {model_path}: {exc}") from exc

    if not hasattr(model, "input_shape") or model.input_shape is None:
        raise ValueError("Loaded model has no valid input_shape")

    expected = (segment_len, 1)
    if model.input_shape[1:] != expected:
        raise ValueError(f"Model expects {model.input_shape[1:]}, script uses {expected}")

    logging.info("Modelis OK, input_shape: %s", model.input_shape)
    return model


def get_ecg_noise_indices_annotated_ext(json_path: Path) -> Optional[List[Tuple[int, int]]]:
    """
    Grąžina [(startIndex, endIndex), ...] iš JSON 'noises_annotated'.
    Jei failo nėra ar raktas neegzistuoja – grąžina None (aiškus signalas).
    """
    if not json_path.exists():
        return None

    with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
        data = json.load(f)

    items = data.get("noises_annotated")
    if not isinstance(items, list):
        return None

    out: List[Tuple[int, int]] = []
    for it in items:
        try:
            out.append((int(it["startIndex"]), int(it["endIndex"])))
        except (KeyError, TypeError, ValueError):
            # praleidžiam brokuotą įrašą, bet netrukdom kitų
            continue
    return out if out else None


def safe_int(value: object) -> Optional[int]:
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def interval_coverage_percent(intervals: Sequence[Tuple[int, int]], total_len: int) -> float:
    """Padengimo dalis % (indeksai laikomi įtrauktiniais: +1)."""
    if total_len <= 0:
        return 0.0
    covered = sum(max(0, end - start + 1) for start, end in intervals)
    return (covered / total_len) * 100.0


def fmt_pct(x: Optional[float | int]) -> str:
    return f"{x:.1f}%" if isinstance(x, (float, int)) else "-"


def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    logging.info("Skaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    filtered = df[df["tag"] != "9999"]
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    logging.info("Atrinkta įrašų: %d", len(names))
    return names, df


# === Pagrindinis srautas ========================================================


def main() -> None:
    setup_logging()

    print("\nĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE", flush=True)
    print("Surandamos išskirtys (outliers), rspragos (rdropouts) ir judesiai (motions - U-Net tipo triukšmai)\n", flush=True)
    logging.info("Filtravimo parametrai: %s", fp)
    logging.info("CONFIG=%s | THRESHOLD=%.3f", CONFIG, THRESHOLD)
    logging.info("Duomenų aplankas: %s | Parametrų aplankas: %s", DUOMENU_APLANKAS, PARAMETRU_APLANKAS)
    logging.info("Model: %s | Noise log: %s.[txt|xlsx]", MODEL_FILE_NAME, NOISE_LOG_BASENAME)
    logging.info("Rezultatų aplankas: %s", REZULTATU_APLANKAS)

    check_consistency(MODEL_FILE_NAME, NOISE_LOG_BASENAME, CONFIG)

    model = load_model_checked(PARAMETRU_APLANKAS / MODEL_FILE_NAME, SEGMENT_LENGTH)
    file_names, df_meta = read_filenames_from_excel(REC_DIR / EXCEL_NAME)

    # Kaupiame rezultatus
    text_log: list[str] = []
    rows_xlsx: list[dict] = []

    total_noise_p = 0.0
    n_files = len(file_names)

    n_annotated_files = 0
    sum_annot_across_annotated = 0.0
    sum_tp_in_annotated_files = 0.0  # bendras tp (algoritmo) tik tuose įrašuose, kurie turi anotacijas

    for i, fname in enumerate(file_names, start=1):
        fpath = REC_DIR / fname

        try:
            ecg = get_ecg_signal(str(fpath))
            annot = get_ecg_noise_indices_annotated_ext(fpath.with_suffix(".json"))
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"Failed to load {fname}: {exc}") from exc

        # Filtravimas
        ecg_f = ecg_filter(ecg, fp)

        # (A) Anotuotų triukšmų padengimas
        if annot is not None:
            n_annotated_files += 1
            merged_ann = merge_lists_of_tuples(annot)
            annot_p = interval_coverage_percent(merged_ann, len(ecg_f))
            sum_annot_across_annotated += annot_p
            line3 = f"Anotuotų triukšmų kiekis: {len(annot)} | Anotuotų triukšmų dalis: {annot_p:.2f}%"
        else:
            annot_p = None
            line3 = "Anotuotų triukšmų nėra"

        # (B) Outliers + rdropouts + U-Net
        ecg_clean, out_idx, rdr_idx = find_outliers_rdropouts(ecg_f)
        _, unet_idx = run_ecg_denoising_pipeline(ecg_clean, model, CONFIG, THRESHOLD)

        merged_all = merge_lists_of_tuples([*unet_idx, *rdr_idx, *out_idx])
        tp = interval_coverage_percent(merged_all, len(ecg_f))
        total_noise_p += tp
        if annot is not None:
            sum_tp_in_annotated_files += tp

        # Meta duomenys iš Excel pagal stem
        stem = fpath.stem
        row = df_meta[df_meta["filename"] == stem].head(1)

        if not row.empty:
            quality = safe_int(row.iloc[0].get("quality"))
            noni = row.iloc[0].get("noni")
            tag = safe_int(row.iloc[0].get("tag"))
            mark = row.iloc[0].get("mark")
            N = row.iloc[0].get("N")
            S = row.iloc[0].get("S")
            V = row.iloc[0].get("V")
            comment = row.iloc[0].get("comment")
            line2 = f"quality:{quality} noni:{noni} tag:{tag} mark:{mark} N:{N} S:{S} V:{V} comment:{comment}"
        else:
            quality = tag = None
            noni = mark = N = S = V = comment = None
            line2 = f"Filename {fname} not found."

        line1 = (
            f"\n{i}. {fname} | outliers:{len(out_idx)} rdropouts:{len(rdr_idx)} "
            f"motions:{len(unet_idx)} | visų triukšmų padengimas: {tp:.2f}%"
        )
        print(line1)
        print(line2)
        if annot is not None:
            print(line3)

        text_log.extend([line1, line2])
        if annot is not None:
            text_log.append(line3)

        rows_xlsx.append(
            {
                "fn": fname,
                "qlt": quality,
                "tag": tag,
                "out": len(out_idx),
                "out%": round(interval_coverage_percent(out_idx, len(ecg_f)), 1),
                "rdr": len(rdr_idx),
                "rdr%": round(interval_coverage_percent(rdr_idx, len(ecg_f)), 1),
                "mot": len(unet_idx),
                "mot%": round(interval_coverage_percent(unet_idx, len(ecg_f)), 1),
                "total%": round(tp, 1),
                "annot": len(annot) if annot is not None else None,
                "annot%": round(annot_p, 1) if annot_p is not None else None,
            }
        )

    # Išsaugojimas: TXT + XLSX
    REZULTATU_APLANKAS.mkdir(parents=True, exist_ok=True)

    noise_txt = REZULTATU_APLANKAS / f"{NOISE_LOG_BASENAME}.txt"
    with open(noise_txt, "w", encoding="utf-8") as f:
        for line in text_log:
            f.write(line + "\n")
    print(f"\nTxt noise_log written to: {noise_txt}")

    df_log = pd.DataFrame(rows_xlsx)
    
    noise_xlsx = REZULTATU_APLANKAS / f"{NOISE_LOG_BASENAME}.xlsx"
    df_log.to_excel(noise_xlsx, index=False)
    print(f"Excel noise_log written to: {noise_xlsx}")

    # Summary
    summary_lines: list[str] = []
    
    summary_lines.append("\nĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE")
    summary_lines.append("Surandamos išskirtys (outliers), rspragos (rdropouts) ir judesiai (motions - unet tipo triukšmai)\n")
    summary_lines.append("PARAMETRAI:")
    summary_lines.append(f"Filtravimo parametrai: {fp}")
    summary_lines.append(f"CONFIG={CONFIG} | THRESHOLD={THRESHOLD:.3f}")
    summary_lines.append(f"Duomenų aplankas: {DUOMENU_APLANKAS} | Parametrų aplankas: {PARAMETRU_APLANKAS}")
    summary_lines.append(f"Model: {MODEL_FILE_NAME} | Noise log: {NOISE_LOG_BASENAME}.[txt|xlsx]")
    summary_lines.append(f"Rezultatų aplankas: {REZULTATU_APLANKAS}")

    summary_lines.append("\nVISŲ ĮRAŠŲ REZULTATAI:")
    summary_lines.append(f"Visų analizuotų įrašų skaičius: {n_files}")

    avg_tp_all = (total_noise_p / n_files) if n_files else 0.0
    summary_lines.append(f"Vidutinis triukšmo padengimas įrašuose: {avg_tp_all:.1f}%")

    summary_lines.append("Vidutinis triukšmo padengimas 'per quality':")
    for q in (0, 1, 2):
        vals = df_log.loc[df_log["qlt"] == q, "total%"]
        avg_q = round(vals.mean(), 1) if not vals.empty else None
        summary_lines.append(f"quality {q}: {fmt_pct(avg_q)}")

    summary_lines.append("\nANOTUOTŲ ĮRAŠŲ REZULTATAI:")
    summary_lines.append(f"Anotuotų įrašų skaičius: {n_annotated_files}")

    avg_annot = (sum_annot_across_annotated / n_annotated_files) if n_annotated_files else 0.0
    avg_tp_in_annot = (
        (sum_tp_in_annotated_files / n_annotated_files) if n_annotated_files else 0.0
    )
    summary_lines.append(f"Vidutinis anotuotų triukšmų padengimas: {avg_annot:.1f}%")
    summary_lines.append(
        "Vidutinis aptiktų triukšmų padengimas anotuotuose įrašuose: "
        f"{avg_tp_in_annot:.1f}%"
    )

    summary_lines.append("\nSUTRUMPINIMAI LENTELĖSE:")
    summary_lines.append("fn    - filename (įrašo pavadinimas)")
    summary_lines.append("qlt   - quality (0,1,2 iš Excel, įrašo kokybė)")
    summary_lines.append("tag   - tag (NA arba 1111, 3333, 9999 iš Excel)")
    summary_lines.append("out   - outliers (išskirtys)")
    summary_lines.append("out%  - outliers padengimo dalis %")
    summary_lines.append("rdr   - rdropouts (rspragos)")
    summary_lines.append("rdr%  - rdropouts padengimo dalis %")
    summary_lines.append("mot   - motions (U-Net)")
    summary_lines.append("mot%  - motions (U-Net) padengimo dalis %")
    summary_lines.append("total%  - visų aptiktų triukšmų padengimo dalis % (out+rdr+mov)")
    summary_lines.append("anot   - anotuoti triukšmai (iš JSON)")
    summary_lines.append("anot%  - anotuotų triukšmų (iš JSON) padengimo dalis %")

    summary_path = REZULTATU_APLANKAS / f"summary{MARKER}.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        for line in summary_lines:
            f.write(line + "\n")

    print(f"Noise_summary written to: {summary_path}")
    print("\n=== APIBENDRINTI REZULTATAI ===")
    with open(summary_path, "r", encoding="utf-8") as f:
        print(f.read())

    
if __name__ == "__main__":
    main()



ĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE
Surandamos išskirtys (outliers), rspragos (rdropouts) ir judesiai (motions - U-Net tipo triukšmai)



13:37:14 | INFO | Filtravimo parametrai: {'type': 'lowpass', 'method': 'butterworth', 'order': 5, 'sampling_rate': 200, 'lowcut': 0.5, 'highcut': 90}
13:37:14 | INFO | CONFIG={'FS': 200, 'SEGMENT_LENGTH': 1024, 'OVERLAP': 0.5} | THRESHOLD=0.080
13:37:14 | INFO | Duomenų aplankas: /home/kestutis/DI/2025_ZIVEO/DUOMENYS_UPD | Parametrų aplankas: /home/kestutis/DI/2025_ZIVEO/DUOMENYS_UPD/PARAMETRAI
13:37:14 | INFO | Model: resunet_ecg_1024_0_5_3_6.keras | Noise log: noise_log_1024_0_5_3_6.[txt|xlsx]
13:37:14 | INFO | Rezultatų aplankas: /home/kestutis/DI/2025_ZIVEO/DUOMENYS_UPD/REZULTATAI
13:37:14 | INFO | Suderinamumas OK | sufiksas:_1024_0_5_3_6 | SEGMENT_LENGTH:1024 | OVERLAP:0_5
13:37:14 | INFO | Kraunamas modelis: /home/kestutis/DI/2025_ZIVEO/DUOMENYS_UPD/PARAMETRAI/resunet_ecg_1024_0_5_3_6.keras
13:37:15 | INFO | Modelis OK, input_shape: (None, 1024, 1)
13:37:15 | INFO | Skaitomas Excel: /home/kestutis/DI/2025_ZIVEO/DUOMENYS_UPD/records_npy_all/visi_zive_irasai_atrankai.xlsx
13:37:15


1. 1000_1.npy | outliers:0 rdropouts:0 motions:0 | visų triukšmų padengimas: 0.00%
quality:0 noni:nan tag:None mark:x N:859 S:2 V:0 comment:Truputi plaukioja amplitudė

2. 1001_1.npy | outliers:0 rdropouts:0 motions:8 | visų triukšmų padengimas: 21.20%
quality:0 noni:nan tag:None mark:nan N:788 S:5 V:0 comment:Yra S. Nedidelis, bet pastovus  BW, yra epizodų su EMG

3. 1001_2.npy | outliers:0 rdropouts:0 motions:4 | visų triukšmų padengimas: 12.00%
quality:0 noni:11 tag:None mark:x N:741 S:4 V:1 comment:izolinija šokinėja nedaug, vietomis išlenda raumenų triušmai
Anotuotų triukšmų kiekis: 11 | Anotuotų triukšmų dalis: 12.27%

4. 1001_3.npy | outliers:0 rdropouts:0 motions:3 | visų triukšmų padengimas: 6.80%
quality:0 noni:4 tag:None mark:nan N:865 S:4 V:0 comment:BW nedidelis, bet nuolatinis, yra ryškių EMG epizodų
Anotuotų triukšmų kiekis: 4 | Anotuotų triukšmų dalis: 6.41%

5. 1001_4.npy | outliers:1 rdropouts:0 motions:0 | visų triukšmų padengimas: 4.51%
quality:0 noni:nan tag:None 